# 05 — Análisis de Video Segmentado

Aplicar el modelo entrenado sobre todos los frames, post-procesar máscaras
y extraer esqueletos y anchos vasculares.

In [ ]:
import os, cv2, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import segmentation_models_pytorch as smp
from skimage.morphology import skeletonize, remove_small_objects
from skimage.measure import label as sk_label, regionprops

DATA_PATH  = Path(os.environ.get("MICROCIRCULATION_DATA", "/content/drive/MyDrive/microcirculation"))
MODEL_DIR  = DATA_PATH / "models"
SEG_DIR    = DATA_PATH / "segmented"
SEG_DIR.mkdir(exist_ok=True)
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CALIB_UM_PX = 1.0  # µm/píxel — ajustar con la calibración real de tu dispositivo


## 5.1 Cargar modelo entrenado

In [ ]:
# Cargar el mejor modelo (U-Net o U-Net++ según resultado NB04)
model = smp.UnetPlusPlus(
    encoder_name="resnet34", encoder_weights=None,
    in_channels=3, classes=1, activation=None
)
model.load_state_dict(torch.load(MODEL_DIR / "unetpp_r34_best.pth",
                                 map_location=DEVICE))
model = model.to(DEVICE).eval()
print("Modelo cargado.")


## 5.2 Predicción sobre un frame

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

INFER_AUG = A.Compose([
    A.Normalize(mean=(0.485,), std=(0.229,)),
    ToTensorV2()
])

def predict_frame(frame_bgr: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    """Devuelve máscara binaria (uint8, 0/255) para un frame BGR."""
    h, w   = frame_bgr.shape[:2]
    resized = cv2.resize(frame_bgr, (512, 512))
    rgb3    = np.stack([resized]*3, axis=-1) if resized.ndim == 2 else resized

    aug    = INFER_AUG(image=rgb3)
    tensor = aug['image'].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logit = model(tensor)
    prob   = torch.sigmoid(logit).squeeze().cpu().numpy()
    mask   = (prob > threshold).astype(np.uint8) * 255
    return cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)


## 5.3 Post-procesado de máscara

In [ ]:
def postprocess_mask(mask: np.ndarray, min_size_px: int = 30) -> dict:
    """
    Dado un mask binario (0/255):
    - Elimina componentes pequeñas (ruido)
    - Calcula esqueleto
    - Mide ancho vascular por sección transversal
    Retorna dict con mask, skeleton, vessel_width_px.
    """
    binary = mask > 127
    cleaned = remove_small_objects(binary, min_size=min_size_px)

    skeleton = skeletonize(cleaned)
    labeled  = sk_label(cleaned)
    props    = regionprops(labeled)

    # Ancho promedio = 2 * sqrt(area/pi) para cada componente (aprox. cilíndrica)
    widths_px = [2 * np.sqrt(p.area / np.pi) for p in props if p.area > min_size_px]

    return {
        "mask_clean"     : cleaned.astype(np.uint8) * 255,
        "skeleton"       : skeleton.astype(np.uint8) * 255,
        "n_vessels"      : len(props),
        "mean_width_px"  : float(np.mean(widths_px)) if widths_px else 0.0,
        "widths_px"      : widths_px
    }


## 5.4 Métricas morfológicas por frame

In [ ]:
def compute_frame_density(mask_clean: np.ndarray, calib: float) -> dict:
    """
    Calcula TVD y PVD a nivel de frame.
    TVD = longitud total de vasos / área del frame (mm/mm²)
    calib = µm/píxel
    """
    skeleton    = skeletonize(mask_clean > 0)
    skel_px     = skeleton.sum()
    h, w        = mask_clean.shape
    area_px2    = h * w

    px_to_mm    = calib / 1000.0                  # µm/px → mm/px
    area_mm2    = area_px2 * (px_to_mm ** 2)
    length_mm   = skel_px * px_to_mm

    TVD = length_mm / area_mm2 if area_mm2 > 0 else 0.0
    return {"TVD_frame": TVD, "skel_px": int(skel_px), "area_mm2": area_mm2}


## 5.5 Procesar todos los videos — guardar máscaras y métricas

In [ ]:
video_files = sorted(DATA_PATH.glob("*.mp4")) + sorted(DATA_PATH.glob("*.avi"))
summary     = []

for vf in video_files:
    vname   = vf.stem
    out_dir = SEG_DIR / vname
    out_dir.mkdir(exist_ok=True)

    cap   = cv2.VideoCapture(str(vf))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    step  = max(1, int(fps))          # 1 frame/segundo para análisis

    frame_metrics = []
    fidx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fidx % step == 0:
            mask   = predict_frame(frame)
            post   = postprocess_mask(mask)
            dens   = compute_frame_density(post["mask_clean"], CALIB_UM_PX)

            # Guardar overlay cada 5 frames
            if (fidx // step) % 5 == 0:
                overlay = frame.copy()
                overlay[post["skeleton"] > 0] = [0, 255, 100]
                cv2.imwrite(str(out_dir / f"frame_{fidx:05d}_overlay.png"), overlay)

            frame_metrics.append({
                "frame_idx" : fidx,
                "n_vessels" : post["n_vessels"],
                "TVD_frame" : dens["TVD_frame"],
                "mean_width": post["mean_width_px"] * CALIB_UM_PX
            })
        fidx += 1
    cap.release()

    fdf = pd.DataFrame(frame_metrics)
    fdf.to_csv(out_dir / "frame_metrics.csv", index=False)

    summary.append({
        "video"     : vname,
        "TVD_auto"  : fdf["TVD_frame"].median(),   # mediana más robusta que media
        "n_vessels_mean": fdf["n_vessels"].mean()
    })
    print(f"{vname}: TVD_auto={fdf['TVD_frame'].median():.2f}  "
          f"vasos/frame={fdf['n_vessels'].mean():.1f}")

summary_df = pd.DataFrame(summary)
summary_df.to_csv(DATA_PATH / "segmentation_summary.csv", index=False)
print("\nSegmentación completa.")
